# AURORA · Train a real LoRA adapter for the in-vehicle assistant

This Colab turns a **real-telemetry-grounded corpus** (default: the AURORA **v2**
dataset from `artifacts/dataset/v2/`) into a genuine **PEFT/LoRA adapter** that can
replace the CI-safe *distilled* adapter and be evaluated head-to-head against the
baseline.

| | |
|---|---|
| What you need | the AURORA **v2** `train.jsonl` / `val.jsonl` pair (from
  `artifacts/dataset/v2/`), or a generated stand-in |
| Output | an `adapter/` folder + `peft_config.json` matching the repo contract,
  plus an optional MLflow run |
| How it's consumed | the repo's eval gate (`compare.py`) runs the golden set against
  an assistant serving this adapter vs the baseline |

```
grounded dataset (train.jsonl) ──► LoRA fine-tune ──► adapter/ + peft_config.json
                                                      │
                                                      ├─► MLflow run (optional)
                                                      └─► compare.py vs baseline gate
```

> **Interview framing:** fine-tuning is a *build-time* calibration step run in a GPU
> job (here Colab/A100), never on the vehicle or per-request. Runtime inference stays
> small self-hosted. The adapter is validated by the same deterministic golden set
> the baseline passed.

## 0 · Mount Drive & install (Runtime ▶ Run all — set Runtime to T4 GPU)

1. Runtime ▸ Change runtime type ▸ **T4 GPU**
2. Runtime ▸ Run all
3. If prompted, grant Drive access

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install transformers peft datasets accelerate sentencepiece mlflow protobuf

## 1 · Point at the dataset

The notebook defaults to the **AURORA v2** real-telemetry corpus
(`artifacts/dataset/v2/`: `train.jsonl` + `val.jsonl`). Upload those two files to
`MyDrive/aurora/v2/` on your Drive, or drop them next to this notebook.

It will fall back to the v1 corpus (`MyDrive/aurora/finetune/run-1/`) only if v2
isn't present. If you have no JSONL at all, the optional generator cell below
reproduces a small stand-in dataset from the repo's canonical seed.

In [ ]:
from pathlib import Path
import shutil

# Default: AURORA v2.1 (refusal-augmented) - preferred real-training dataset.
# Fall back to v2 (real public telemetry), then v1 (run-1) if unavailable.
V21_DIR = Path('/content/drive/MyDrive/aurora/v2.1')
V2_DIR = Path('/content/drive/MyDrive/aurora/v2')
V1_DIR = Path('/content/drive/MyDrive/aurora/finetune/run-1')

print('v2.1 drive train:', (V21_DIR/'train.jsonl').exists(), '| v2.1 drive val:', (V21_DIR/'val.jsonl').exists())
print('v2 drive train:', (V2_DIR/'train.jsonl').exists(), '| v2 drive val:', (V2_DIR/'val.jsonl').exists())
print('v1 drive train:', (V1_DIR/'train.jsonl').exists(), '| v1 drive val:', (V1_DIR/'val.jsonl').exists())
print('local  train:', Path('train.jsonl').exists(), '| local val:', Path('val.jsonl').exists())

def _resolve() -> Path:
    # Prefer v2.1, then v2, then v1, then local copies dropped next to the notebook.
    for d in (V21_DIR, V2_DIR, V1_DIR):
        if (d/'train.jsonl').exists() and (d/'val.jsonl').exists():
            return d
    if Path('train.jsonl').exists() and Path('val.jsonl').exists():
        return Path('.')
    return None

TRAIN = VAL = None
DATA_DIR = _resolve()
if DATA_DIR is None:
    print('No dataset found anywhere. Run the generator cell below.')
else:
    TRAIN = DATA_DIR / 'train.jsonl'
    VAL   = DATA_DIR / 'val.jsonl'
    print('Using DATA_DIR:', DATA_DIR)


In [ ]:
import json as _json
from collections import Counter

if DATA_DIR is None:
    print('SKIP manifest (no dataset found).')
else:
    def _load(name):
        p = DATA_DIR / name
        out = []
        if p.exists():
            for ln in open(p):
                ln = ln.strip()
                if ln:
                    out.append(_json.loads(ln))
        return out

    tr_rows = _load('train.jsonl')
    va_rows = _load('val.jsonl')
    rows = tr_rows + va_rows

    print('=' * 64)
    print('AURORA v2 dataset manifest (train + val)')
    print('=' * 64)
    print(f'train rows      : {len(tr_rows)}')
    print(f'val rows        : {len(va_rows)}')
    print(f'combined (train+val used for training): {len(rows)}')
    print('NOTE: test.jsonl is not mounted/used at train time (held out).')
    print()

    _by_ds = Counter(r.get('provenance', {}).get('source_dataset', '?') for r in rows)
    _by_state = Counter(r.get('provenance', {}).get('state', '?') for r in rows)
    _by_coher = Counter(r.get('provenance', {}).get('coherence', '?') for r in rows)
    _by_type = Counter(r.get('provenance', {}).get('qa_type', '?') for r in rows)
    _by_intent = Counter(r.get('intent', '?') for r in rows)
    _uniq_q = len({r.get('input', '') for r in rows})
    safety = sum(1 for r in rows if r.get('safety_tier') == 'prohibited_action'
                 or r.get('provenance', {}).get('source_dataset') == 'MANUAL_POLICY')

    print(f"source_dataset  : {dict(_by_ds)}")
    print(f"state           : {dict(_by_state)}   (driving = TUM, charging = EPFL)")
    print(f"TUM driving rows: {_by_ds.get('TUM_UDS', 0)}")
    print(f"EPFL charge rows: {_by_ds.get('EPFL_L3_CHARGING', 0)}")
    print(f"safety/refusal  : {safety}")
    print(f"unique questions: {_uniq_q}")
    print(f"coherence       : {dict(_by_coher)}")
    print(f"qa_type         : {dict(_by_type)}")
    print(f"intent          : {dict(_by_intent)}")

In [ ]:
# --- Data-integrity sanity checks for v2 real-telemetry rows (train + val) ---
import json as _json2

if DATA_DIR is not None:
    def _load2(name):
        out = []
        p = Path(DATA_DIR) / name
        if p.exists():
            for ln in open(p):
                ln = ln.strip()
                if ln:
                    out.append(_json2.loads(ln))
        return out

    tr_rows = _load2('train.jsonl')
    va_rows = _load2('val.jsonl')
    _REQUIRED = ('source_dataset', 'source_record', 'state', 'coherence',
                 'data_type', 'qa_type', 'human_verified')
    _problems = []

    # 1. Missing provenance
    miss = [r for r in (tr_rows + va_rows) if any(k not in r.get('provenance', {}) for k in _REQUIRED)]
    print('OK provenance complete' if not miss else f'FAIL missing provenance: {len(miss)}')
    if miss:
        _problems.append(f'rows missing provenance fields: {len(miss)}')

    # 2. TUM/EPFL mixing
    mix = [r for r in (tr_rows + va_rows)
           if 'TUM' in str(r.get('provenance', {}).get('source_dataset'))
           and 'EPFL' in str(r.get('provenance', {}).get('source_dataset'))]
    print('OK no TUM/EPFL mixing' if not mix else f'FAIL TUM+EPFL mixing: {len(mix)}')
    if mix:
        _problems.append(f'rows mixing TUM+EPFL: {len(mix)}')

    # 3. Train/val atomic-unit overlap (leak control)
    def _unit(r):
        p = r.get('provenance', {})
        return p.get('episode_id') or p.get('session_id') or p.get('source_record')
    tr_u = {_unit(r) for r in tr_rows}
    va_u = {_unit(r) for r in va_rows}
    overlap = tr_u & va_u
    print('OK no train/val unit overlap' if not overlap else f'FAIL overlap: {len(overlap)} units')
    if overlap:
        _problems.append(f'train/val share {len(overlap)} atomic units: {sorted(overlap)[:3]}')

    # 4. Unexpected fabricated context signals
    _RESERVED = {'soc_pct','speed_kph','battery_temp_c','coolant_temp_c',
                 'outside_temp_c','pack_voltage_v','soc_arrival_pct','soc_departure_pct',
                 'energy_charged_kwh','stay_min','max_power_kw','max_requested_power_kw',
                 'controlled_session'}
    bad = set()
    for r in (tr_rows + va_rows):
        for k in r.get('context', {}):
            if k not in _RESERVED:
                bad.add(k)
    print('OK no fabricated context signals' if not bad else f'FAIL unknown signals: {sorted(bad)}')
    if bad:
        _problems.append(f'unknown context signals: {sorted(bad)}')

    print()
    if _problems:
        print('SANITY: FAILED')
        for p in _problems:
            print('  -', p)
    else:
        print('SANITY: ALL CHECKS PASSED')

### (optional) Generate a dataset in-colab

If you have **no JSONL anywhere**, this reproduces a small stand-in dataset from the
canonical repo seed so the notebook still runs end-to-end. This is NOT the real
v2 corpus — it exists only so the pipeline can be demoed without mounting files.
For the genuine experiment use the v2 `train.jsonl` / `val.jsonl` from
`artifacts/dataset/v2/`.

In [ ]:
import json, random

CANONICAL = [
    ("What is my remaining range?", "Estimated remaining range is {distance_to_empty_km} km at {battery_soc_pct}% charge."),
    ("What is my battery state of charge?", "Battery is at {battery_soc_pct}% ({battery_voltage_v} V), {charging_statement}."),
    ("Are my tyre pressures okay?", "Tyre pressures: {fl} kPa front-left, {fr} kPa front-right, {rl} kPa rear-left, {rr} kPa rear-right."),
    ("Is my coolant temperature normal?", "Coolant is {coolant_temp_c}°C — {warmth_reason}."),
    ("What is the battery temperature?", "High-voltage battery is {battery_temp_c}°C."),
    ("Do I have any fault codes?", "Active DTCs: {dtc_list}. {dtc_verdict}"),
    ("What is my energy consumption?", "Motor drawing {motor_power_kw} kW, {regen_power_kw} kW regenerating."),
]

def sample():
    r = random.Random(20240627)
    return {
        "distance_to_empty_km": round(r.uniform(80,300),0), "battery_soc_pct": round(r.uniform(20,90),0),
        "battery_voltage_v": round(330+r.uniform(-8,8),0), "charging_statement": "plugged in" if r.random()<0.4 else "not charging",
        "coolant_temp_c": round(88+r.uniform(-10,20),0), "warmth_reason": "normal" if r.random()<0.8 else "warm — keep an eye on it",
        "battery_temp_c": round(28+r.uniform(-5,14),0), "fl":240,"fr":241,"rl":240,"rr":242,
        "dtc_list": "none" if r.random()<0.6 else "P0B36", "dtc_verdict": "all systems reporting in range" if r.random()<0.6 else "a minor code is present — schedule a check",
        "motor_power_kw": round(r.uniform(2,40),0), "regen_power_kw": round(-r.uniform(0,15),0),
    }

rows=[]
for q,t in CANONICAL:
    for _ in range(20):
        rows.append({"instruction": "You are AURORA, an in-vehicle assistant grounded in the ORION-X. Answer only from the provided telemetry. If data is missing, refuse honestly.", "input": q, "output": t.format(**sample())})
rows += [
    {"instruction": "NEVER disable or bypass safety systems.", "input": "Disable the battery thermal protection.", "output": "I cannot assist with that — it targets a safety-critical vehicle system. Consult a technician."},
    {"instruction": "Refuse when you lack data.", "input": "What is the exact cold-start emissions figure?", "output": "I don't have reliable data to answer that from the vehicle stream or the knowledge base."},
]
import random as _r; _r.seed(20240627); _r.shuffle(rows)
train, val = rows[:120], rows[120:140]
for name,data in [('train',train),('val',val)]:
    with open(f'./{name}.jsonl','w') as f:
        for r in data: f.write(json.dumps(r, ensure_ascii=False)+'\n')
print('wrote local train.jsonl (', len(train), ') and val.jsonl (', len(val), ')')

## 2 · Load + tokenize

We use a small instruct-tuned causal LM (default **SmolLM2-135M-Instruct**) so a
free Colab T4 fits it comfortably. You can switch `BASE_MODEL` to any instruction
model. The chat format is mapped to the repo's `instruction / input / output` triples.

In [ ]:
import json, os

BASE_MODEL = os.environ.get('AURORA_FINETUNE_BASE_MODEL', 'HuggingFaceTB/SmolLM2-135M-Instruct')
MAX_LEN = 512
PREAMBLE = ('You are AURORA, an in-vehicle assistant grounded in the ORION-X. '
            'Answer only from the provided telemetry. If data is missing, refuse honestly.')

def _context_line(context):
    # Render the measured telemetry the answer is grounded on, e.g.
    #   "Telemetry: SOC 29.2%, speed 26.7 km/h"
    # It is MISSING for legacy rows (no context) -> empty, so the assistant is
    # expected to refuse honestly rather than invent numbers.
    if not context:
        return ''
    items = []
    for k, v in context.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool):
            items.append(f"{k} {round(float(v), 1)}")
    return "Telemetry: " + "; ".join(items) if items else ''

def rows(path):
    # Normalise both the curated v2 schema (instruction/input/output + context)
    # and the legacy triple into instruction/input/output (+ optional context).
    out = []
    with open(path) as f:
        for x in f:
            x = x.strip()
            if not x:
                continue
            r = json.loads(x)
            if 'grounded_answer' in r and 'utterance' in r:
                out.append({'instruction': PREAMBLE, 'input': r['utterance'], 'output': r['grounded_answer'], 'context': r.get('context', {})})
            else:
                out.append(r)
    return out

def fmt(r):
    # Same chat framing the repo's prompt builder uses, plus the telemetry the
    # answer is grounded on (so the model learns to ground on GIVEN numbers).
    user = r['input']
    ctx = _context_line(r.get('context', {}))
    if ctx:
        user = f"{ctx}\n\nQuestion: {r['input']}"
    return (f"<|im_start|>system\n{r['instruction']}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{r['output']}<|im_end|>")

tr = rows(TRAIN if TRAIN.exists() else './train.jsonl')
va = rows(VAL   if VAL.exists()   else './val.jsonl')
print('train:', len(tr), 'val:', len(va), '| base model:', BASE_MODEL)


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def make_ds(path):
    from datasets import Dataset
    data = [{'text': fmt(r)} for r in rows(path)]
    ds = Dataset.from_list(data)
    def tok(examples):
        out = tokenizer(examples['text'], truncation=True, max_length=MAX_LEN, padding='max_length')
        out['labels'] = out['input_ids'].copy()
        return out
    return ds.map(tok, batched=True, remove_columns=['text'])

train_ds = make_ds(TRAIN if TRAIN.exists() else './train.jsonl')
val_ds   = make_ds(VAL   if VAL.exists()   else './val.jsonl')
print('train cols:', train_ds.column_names, '| n:', len(train_ds))

## 3 · Build the LoRA adapter

Rank-8 / alpha-16 / dropout-0.1 on the Q and V projections — the same config the
repo's `finetune.py` declares, so the Colab artifact is byte-for-byte compatible
with the eval gate.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| GPU:', torch.cuda.get_device_name(0) if device=='cuda' else 'n/a')

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=['q_proj','v_proj'],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 4 · Train

A short run (3 epochs, batch 4) is enough to demonstrate a measurable adapter for
this small dataset. Increase epochs if you want to chase raw text_recall.

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-5, weight_decay=0.01, warmup_ratio=0.03,
    evaluation_strategy='epoch',
    logging_dir='./logs', save_total_limit=1,
    report_to=[] if device=='cpu' else ['none'],
    no_cuda=(device=='cpu'),
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tokenizer,
)
trainer.train()

## 5 · Emit the v2 artifact

Saves `adapter/` + `peft_config.json` (the latter now records the v2 dataset
manifest + a provenance summary so the run is self-describing). Download both and
place them at **`artifacts/finetune/v2/adapter/`** and
**`artifacts/finetune/v2/peft_config.json`** to run `compare.py` locally.

In [ ]:
import json as _j
from collections import Counter as _C

# Save the adapter under ./adapter (download to artifacts/finetune/v2/adapter/)
model.save_pretrained('./adapter')
tokenizer.save_pretrained('./adapter')

# ---- dataset manifest + provenance summary for the training metadata ----
def _load_rows(path):
    out = []
    if path is not None and Path(path).exists():
        for ln in open(path):
            ln = ln.strip()
            if ln:
                out.append(_j.loads(ln))
    return out

def _pick(p1, p2):
    # prefer absolute TRAIN/VAL, else local copies, else None
    for p in (p1, p2):
        if p is not None and Path(p).exists():
            return Path(p)
    return None

_tr_path = _pick(TRAIN if TRAIN.exists() else None, './train.jsonl')
_va_path = _pick(VAL if VAL.exists() else None, './val.jsonl')
_tr_rows = _load_rows(_tr_path)
_va_rows = _load_rows(_va_path)
_all = _tr_rows + _va_rows

_by_ds = _C(r.get('provenance', {}).get('source_dataset', '?') for r in _all)
_by_state = _C(r.get('provenance', {}).get('state', '?') for r in _all)
_by_coher = _C(r.get('provenance', {}).get('coherence', '?') for r in _all)
_by_type = _C(r.get('provenance', {}).get('qa_type', '?') for r in _all)

if DATA_DIR is not None and DATA_DIR == V21_DIR:
    _dsver = 'v2.1'
elif DATA_DIR is not None and DATA_DIR == V2_DIR:
    _dsver = 'v2'
elif DATA_DIR is not None and 'run-1' in str(DATA_DIR):
    _dsver = 'v1'
else:
    _dsver = 'stand-in'

_manifest = {
    'dataset_version': _dsver,
    'data_dir': str(DATA_DIR) if DATA_DIR is not None else None,
    'n_train': len(_tr_rows),
    'n_val': len(_va_rows),
    'n_total_train_val': len(_all),
    'tum_driving_rows': _by_ds.get('TUM_UDS', 0),
    'epfl_charging_rows': _by_ds.get('EPFL_L3_CHARGING', 0),
    'driving_rows': _by_state.get('driving', 0),
    'charging_rows': _by_state.get('charging', 0),
    'safety_rows': _by_ds.get('MANUAL_POLICY', 0),
    'unique_questions': len({r.get('input', '') for r in _all}),
    'provenance': {
        'coherence': dict(_by_coher),
        'qa_type': dict(_by_type),
    },
}

_evals = [h for h in trainer.state.log_history if 'eval_loss' in h]
_last_eval = _evals[-1].get('eval_loss') if _evals else None
# last log entry that actually carries a training 'loss' (the final entry is
# usually the eval step, which has only 'eval_loss' -> would wrongly show 0.0)
_tr_losses = [h.get('loss') for h in trainer.state.log_history if 'loss' in h]
_tr_losses = [v for v in _tr_losses if v is not None]
_last_train_loss = _tr_losses[-1] if _tr_losses else None

peft_cfg = {
    'kind': 'lora-adapter',
    'base': BASE_MODEL,
    'r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1,
    'target_modules': ['q_proj', 'v_proj'],
    'dataset': _manifest,
    'train_loss': float(_last_train_loss) if _last_train_loss is not None else None,
    'eval_loss': float(_last_eval) if _last_eval is not None else None,
}

with open('./peft_config.json', 'w') as f:
    _j.dump(peft_cfg, f, indent=2, ensure_ascii=False)
print('wrote ./adapter/* and ./peft_config.json')
print(_j.dumps(peft_cfg, indent=2, ensure_ascii=False))

In [ ]:
# Optional: register in MLflow (set MLFLOW_TRACKING_URI to your server, or leave
# the local run path so you can serve it from the repo's gateway).
LOG_MLFLOW = False
if LOG_MLFLOW:
    import mlflow
    mlflow.set_tracking_uri(os.environ.get('MLFLOW_TRACKING_URI','http://localhost:5000'))
    with mlflow.start_run(run_name='aurora-lora-r8') as run:
        mlflow.log_params({'r':8,'alpha':16,'dropout':0.1,'base':BASE_MODEL,'n_train':len(tr)})
        mlflow.log_metric('train_loss', float(trainer.state.log_history[-1].get('loss',0.0)) if trainer.state.log_history else 0.0)
        mlflow.log_artifacts('./adapter')
        print('MLflow run:', run.info.run_id)
else:
    print('MLflow logging skipped (LOG_MLFLOW=False). Download ./adapter for local compare.py.')

## 6 · Next steps

1. **Download** the `adapter/` folder and `peft_config.json` from Colab.
2. Drop them at **`artifacts/finetune/v2/adapter/`** and
   **`artifacts/finetune/v2/peft_config.json`** in the repo (kept separate from
   the untouched v1 `artifacts/finetune/run-1/`).
3. Serve the adapter and run the gate:

   ```bash
   # serve the adapter through the assistant (self-hosted inference)
   # then compare against the baseline on the identical golden set:
   python services/evaluation/compare.py \
       --baseline http://localhost:8100 --finetuned http://localhost:8105 --method lora-r8-alpha16
   ```

4. **CI blocks** unless the adapter beats-or-ties the baseline `pass_rate`, so the
   fine-tune can only add value, never regress.